## **Previsão da Vida Útil Remanescente de Motores *Turbofan* com Validação Agrupada por Unidade no Dataset C-MAPSS**
 
**Base de dados:** C-MAPSS — Commercial Modular Aero-Propulsion System Simulation

Este notebook apresenta uma análise inicial relacionada ao desenvolvimento experimental do plano de tese, com ênfase na manutenção preditiva fundamentada na estimação da Vida Útil Remanescente (RUL) de motores turbofan. Nesta etapa, busca-se organizar e compreender o conjunto de dados C-MAPSS, estruturar o pipeline de pré-processamento, construir os rótulos de RUL e estabelecer as primeiras baselines determinísticas de aprendizado de máquina. Assim, este notebook está relacionado com as etapas iniciais do plano de trabalho, especialmente à organização dos dados, à preparação do pipeline experimental e à implementação de modelos de referência.

### **1. O Dataset C-MAPSS**

#### **1.1 O que é o C-MAPSS?**

O **C-MAPSS** (*Commercial Modular Aero-Propulsion System Simulation*) é uma simulação computacional desenvolvida pela NASA para representar o comportamento de um motor turbofan comercial. Conforme descrito por Liu et al. (2012), o C-MAPSS é um modelo **não linear**, **dinâmico** e em **nível de componentes** (*Component-Level Model — CLM*) de um motor turbofan comercial de **alto bypass** e **duplo eixo** (*dual-spool*). O modelo foi implementado em ambiente MATLAB/Simulink e permite simular o motor, seu sistema de controle e diferentes condições de operação.

Em termos práticos, o C-MAPSS pode ser entendido como um ambiente de simulação no qual um motor de aeronave é representado matematicamente por seus principais componentes físicos. Segundo Liu et al. (2012), o motor modelado possui componentes como **entrada de ar**, **fan**, **compressor de baixa pressão (LPC)**, **compressor de alta pressão (HPC)**, **combustor**, **turbina de alta pressão (HPT)**, **turbina de baixa pressão (LPT)** e **bocal de saída**. O documento também descreve que a HPT fornece potência ao HPC, enquanto a LPT aciona tanto o LPC quanto o fan.

Dessa forma, cada linha do dataset representa uma condição de operação simulada de um motor turbofan ao longo do tempo, com medições associadas ao funcionamento interno do sistema. O objetivo do conjunto de dados é permitir o estudo de **degradação progressiva** e **previsão de falha** em motores aeronáuticos.

#### **1.2 O que o dataset representa?**

Os dados são compostos por **múltiplas séries temporais multivariadas**. Cada série temporal corresponde a um motor diferente. Assim, o conjunto de dados pode ser interpretado como uma frota de motores do mesmo tipo, em que cada unidade apresenta sua própria trajetória de funcionamento até a falha ou até um ponto anterior à ela.

Cada motor inicia sua trajetória em condição operacional normal. Entretanto, ao longo dos ciclos, uma falha passa a se desenvolver em determinado momento. No conjunto de **treinamento**, essa falha cresce progressivamente até atingir a falha final do sistema. No conjunto de **teste**, a trajetória é interrompida antes da falha final, sendo necessário prever quantos ciclos operacionais ainda restam até a falha. Esse número de ciclos restantes é chamado de **RUL** (*Remaining Useful Life*), ou **vida útil remanescente**.

Portanto, o problema associado ao dataset pode ser resumido da seguinte forma:

> Dado o histórico de operação e os sensores de um motor turbofan, deseja-se estimar quantos ciclos operacionais ainda restam antes da falha.

Dessa forma, o C-MAPSS é adequado para a análise de **manutenção preditiva**, pois o objetivo não é apenas identificar que uma falha já ocorreu, mas também antecipá-la com base nos sinais progressivos de degradação.

#### **1.3 Como o motor é representado na simulação?**

De acordo com Liu et al. (2012), o motor do C-MAPSS é representado como um sistema dinâmico não linear com duas variáveis de estado principais: a **velocidade do fan** (*fan speed*) e a **velocidade do núcleo** (*core speed*). O modelo é construído em nível de componentes: cada componente do motor é modelado separadamente e, em seguida, conectado aos demais componentes para compor a simulação completa.

O autor informa ainda que o desempenho dos componentes é calculado por meio de relações termodinâmicas e de interpolação a partir de mapas de desempenho do fan, dos compressores e das turbinas. Além disso, o modelo inclui parâmetros de saúde do motor, utilizados para simular degradação. Esses parâmetros de saúde representam modificadores de fluxo, da eficiência e da razão de pressão em componentes como o fan, o LPC, o HPC, o HPT e o LPT.

Isso significa que a degradação simulada não é adicionada como um rótulo artificial sem relação com o motor. Ela é incorporada ao comportamento físico do sistema por meio de alterações nos parâmetros internos de saúde. Como consequência, os sensores passam a refletir mudanças no desempenho do motor à medida que a degradação progride.

#### **1.4 Estrutura geral dos arquivos do dataset**

O dataset é dividido em subconjuntos identificados como **FD001**, **FD002**, **FD003** e **FD004**. Cada subconjunto possui três tipos principais de arquivo:

| Tipo de arquivo | Exemplo | Função no projeto |
|---|---|---|
| Treinamento | `train_FD001.txt` | Contém trajetórias de motores até a falha. É usado para treinar os modelos. |
| Teste | `test_FD001.txt` | Contém trajetórias interrompidas antes da falha. É usado para gerar previsões. |
| Gabarito do teste | `RUL_FD001.txt` | Contém o RUL real dos motores de teste, usado para avaliar as previsões. |

Cada linha do arquivo representa um **ciclo operacional** de um motor. As colunas correspondem a:

| Grupo de colunas | Significado |
|---|---|
| `unit` | Identificador do motor. |
| `cycle` | Tempo em ciclos operacionais. |
| `op1`, `op2`, `op3` | Configurações operacionais que influenciam o desempenho do motor. |
| `s1` a `s21` | Medições de sensores do motor. |

Assim, uma linha do dataset pode ser lida como: “no ciclo `t`, o motor `i` operou sob determinadas condições operacionais e apresentou determinados valores nos sensores”.

#### **1.5 Diferença entre treinamento, teste e RUL**

No arquivo de **treinamento**, cada motor é acompanhado até a falha. Por esse motivo, o RUL pode ser calculado diretamente a partir do último ciclo observado de cada motor. Se o motor $i$ apresenta falha no ciclo final $T_i$, então o RUL no ciclo atual $t$ é dado por:

$$
\operatorname{RUL}_i(t) = T_i - t
$$

em que $\operatorname{RUL}_i(t)$ representa a vida útil remanescente do motor $i$ no ciclo $t$, $T_i$ representa o último ciclo observado desse motor no conjunto de treinamento e $t$ representa o ciclo atual.

No arquivo de **teste**, a falha ainda não ocorreu na sequência observada. A série temporal termina antes do ciclo de falha. Por isso, o arquivo `RUL_FDxxx.txt` fornece o número real de ciclos restantes após o último ciclo observado no teste. O objetivo do modelo é prever esse valor.

#### **1.6 Subconjuntos FD001, FD002, FD003 e FD004**

Os quatro subconjuntos diferem quanto ao número de condições operacionais e ao número de modos de falha:

| Subconjunto | Condições operacionais | Modos de falha | Caracterização |
|---|---:|---:|---|
| FD001 | 1 | 1 | Cenário mais simples: uma condição operacional e degradação no HPC. |
| FD002 | 6 | 1 | Um modo de falha, porém com múltiplas condições operacionais. |
| FD003 | 1 | 2 | Uma condição operacional, porém com dois modos de falha. |
| FD004 | 6 | 2 | Cenário mais complexo: múltiplas condições operacionais e múltiplos modos de falha. |

O subconjunto FD001 foi selecionado como estudo de caso principal por permitir a análise da degradação de motores turbofan em um cenário operacional controlado. Por apresentar uma única condição de operação e um único modo de falha, esse subconjunto favorece a identificação dos padrões de degradação presentes nos sensores, sem a necessidade inicial de modelar múltiplos regimes operacionais ou mecanismos de falha simultâneos. Assim, sua utilização permite avaliar objetivamente a capacidade dos modelos de aprendizado de máquina de estimar a vida útil remanescente a partir de séries temporais multivariadas.

#### **1.7 Por que esse dataset é adequado para manutenção preditiva?**

O C-MAPSS é adequado para manutenção preditiva porque representa a evolução temporal de motores até a falha ou até um ponto anterior a ela. Como há medições de sensores ao longo dos ciclos, é possível estudar como os sinais operacionais se alteram à medida que o motor se degrada.

Em um cenário de manutenção corretiva, o equipamento é reparado apenas após falhar. Em um cenário de manutenção preventiva, a intervenção ocorre em intervalos fixos. Já na manutenção preditiva, busca-se utilizar dados de operação para estimar a condição do equipamento e antecipar a falha. O C-MAPSS permite desenvolver esse tipo de abordagem, pois fornece trajetórias de funcionamento, sensores, condições operacionais e valores de RUL para avaliação.



### **2. Uso de Aprendizado de Máquina com o C-MAPSS na Previsão de RUL**

O C-MAPSS serve como base experimental para estudos de **manutenção preditiva** e de **prognóstico de falhas**, pois permite tratar a degradação de motores turbofan como um problema supervisionado de previsão da **vida útil remanescente** (*Remaining Useful Life — RUL)*. Nessa abordagem, as leituras de sensores e as condições operacionais observadas ao longo dos ciclos constituem as variáveis de entrada, e o RUL é a saída a ser estimada.

A literatura recente mostra que o uso de aprendizado de máquina nesse *dataset* pode ser organizado em três linhas principais: modelos clássicos de regressão, modelos baseados em séries temporais e abordagens voltadas à interpretabilidade e à incerteza das previsões. Essas linhas não são excludentes. Pelo contrário, elas formam uma sequência natural de amadurecimento experimental, em que a transição ocorre da seguinte forma: primeiro, são construídos modelos determinísticos de referência; em seguida, exploram-se modelos capazes de representar dependências temporais; por fim, investigam-se explicações, a confiabilidade e a incerteza associadas às previsões.

#### **2.1 Modelos clássicos de regressão como referência experimental**

Uma primeira forma de tratar o C-MAPSS consiste em transformar cada ciclo de operação em uma amostra supervisionada, utilizando os sensores e as variáveis operacionais como atributos preditores. Nesse contexto, o problema passa a ser tratado como uma tarefa de **regressão**, pois o objetivo é estimar um valor numérico contínuo: o número de ciclos restantes até a falha.

Jia, Xiao e Shi (2021) aplicaram ao C-MAPSS modelos como **Decision Tree**, **Random Forest**, **Support Vector Regression** e **XGBoost** após as etapas de pré-processamento e de seleção de atributos. Os autores destacam o XGBoost como uma abordagem de *ensemble* capaz de combinar sucessivos modelos fracos em um preditor mais forte, alcançando desempenho competitivo em relação a modelos tradicionais. De forma semelhante, Goyal e Sachdeva (2025) compararam **SVM**, **Random Forest** e **XGBoost** no C-MAPSS, utilizando métricas como **MAE** e **RMSE**, e observaram desempenho superior dos modelos baseados em árvores, especialmente Random Forest e XGBoost.

Esses estudos justificam o uso inicial de modelos clássicos neste notebook. Modelos como **Random Forest**, **Gradient Boosting** e **XGBoost** são adequados para uma primeira etapa de modelagem porque lidam bem com relações não lineares entre sensores e RUL, permitem comparar desempenho por métricas objetivas e oferecem algum grau de interpretação por meio da importância das variáveis. Além disso, modelos lineares e métodos como SVR podem ser utilizados como referências de comparação, desde que o pré-processamento seja conduzido adequadamente, especialmente a normalização dos atributos.

#### **2.2 Séries temporais, janelamento e modelos de Deep Learning**

Embora a formulação por ciclo seja útil para estabelecer *baselines*, o C-MAPSS possui natureza essencialmente temporal. Cada motor é acompanhado ao longo de uma sequência de ciclos, e a degradação não se manifesta apenas em uma leitura isolada, mas também na evolução dos sensores ao longo do tempo. Por isso, uma parte significativa da literatura trata a previsão de RUL como um problema de **séries temporais multivariadas**.

Khalifeh e AlMeqdadi (2024) investigaram o impacto do tamanho da janela temporal na estimação de RUL com **LSTM** e **CNN**, mostrando que a forma de segmentação dos dados influencia diretamente o desempenho dos modelos. Janelas maiores tendem a favorecer modelos recorrentes, como LSTM, por preservarem dependências temporais mais longas, enquanto janelas menores podem beneficiar arquiteturas convolucionais ao captar padrões locais nos sensores. 

Chola et al. (2024) também compararam abordagens como **LSTM**, **XGBoost** e **LightGBM** no contexto do C-MAPSS, avaliando tanto o erro de previsão quanto aspectos computacionais. Já Hatipoğlu e Yılmaz (2026) avançaram na direção de arquiteturas profundas com mecanismos de atenção, combinando LSTM/BiLSTM com descritores matriciais calculados a partir de janelas multivariadas dos sensores. Esses trabalhos indicam que modelos temporais e arquiteturas profundas podem capturar padrões mais ricos de degradação, especialmente quando o histórico recente dos sensores é representado explicitamente.

No contexto dessa pesquisa, primeiro será construído um pipeline determinístico e reprodutível com modelos clássicos de regressão. Em seguida, os mesmos dados poderão ser reorganizados em janelas temporais para permitir experimentos com modelos capazes de explorar dependências sequenciais, como CNN 1D, LSTM ou arquiteturas híbridas.

#### **2.3 Interpretabilidade, seleção de variáveis e confiança nas previsões**

Em aplicações de manutenção preditiva, obter baixo erro de previsão não é suficiente. Como o resultado pode orientar decisões de manutenção em sistemas críticos, é necessário compreender quais variáveis influenciam a previsão e se o comportamento do modelo é coerente com os sinais de degradação observados nos sensores.

Avsar (2026) investigou a interpretabilidade de modelos prognósticos no C-MAPSS por meio da aplicação de **SHAP** em conjunto com métricas prognósticas tradicionais, como monotonicidade, *trendability* e prognosticabilidade. Enquanto essas métricas permitem avaliar se os sensores apresentam comportamento compatível com degradação progressiva, o SHAP possibilita analisar a importância efetivamente atribuída às variáveis após o treinamento dos modelos. Dessa forma, o estudo comparou a relevância dos sensores segundo critérios tradicionais de prognóstico com a importância aprendida por modelos como Random Forest e Gradient Boosting. Essa abordagem é relevante porque aproxima o desempenho preditivo da interpretação técnica, permitindo verificar se o modelo utiliza variáveis coerentes com a evolução física da degradação.

Outro ponto relevante é a incerteza. Zhan et al. (2023) observam que grande parte dos estudos de RUL concentra-se em estimativas pontuais, embora o processo de previsão esteja sujeito a incertezas decorrentes do ruído dos sensores, da qualidade dos dados e das limitações do próprio modelo. Os autores propõem uma abordagem de predição intervalar para capturar incertezas de dados e de modelo, validada em subconjuntos do C-MAPSS. Esse tipo de contribuição evidencia que, em etapas futuras, a previsão de RUL pode ser tratada não apenas como um único número, mas também como uma estimativa acompanhada de um intervalo ou de uma medida de confiança.

#### **2.4 Análise a ser realizada neste notebook**

Com base nas referências analisadas, a análise neste notebook será realizada progressivamente. Primeiro, serão realizados o entendimento do conjunto de dados, a criação da variável RUL, a análise exploratória e o pré-processamento. Em seguida, serão treinados modelos de regressão determinísticos para servir como primeira base de comparação. Essa ordem permite compreender melhor o comportamento dos sensores, identificar variáveis relevantes e avaliar os erros dos modelos antes de avançar para abordagens mais complexas.

Os modelos clássicos serão usados como referência inicial de comparação. Com eles, será possível verificar quais sensores parecem mais relacionados à degradação, qual nível de erro é obtido com modelos tabulares e como escolhas de pré-processamento, como normalização e remoção de sensores constantes, influenciam os resultados. Essa etapa também ajuda a identificar os limites da abordagem inicial, principalmente por ainda não explorar toda a estrutura temporal dos dados. A partir dessa base, o trabalho poderá avançar para janelas temporais, modelos de Deep Learning, interpretabilidade e, em etapas posteriores, quantificação de incerteza.


### **3. Preparação do Ambiente e Carregamento dos Dados**

Nesta etapa, importam-se as bibliotecas necessárias, define-se o diretório do conjunto de dados e carregam-se os arquivos do subconjunto FD001. A organização inicial dos dados é importante para garantir que as colunas estejam corretamente nomeadas e que os arquivos de treinamento, de teste e de RUL sejam lidos de forma padronizada.


#### **3.1 Importação das bibliotecas**

As bibliotecas utilizadas nesta etapa são voltadas à manipulação de dados, operações numéricas, visualização gráfica e organização de caminhos de arquivo. Outras bibliotecas específicas de modelagem serão importadas em seções posteriores, se necessárias.


In [2]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


In [3]:
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

#### **3.2 Definição do diretório dos dados**

Os arquivos do C-MAPSS precisam estar no caminho: `data/raw/CMAPSSData`. 

In [5]:
DATA_DIR = Path("data/raw/CMAPSSData")

if not DATA_DIR.exists():
    raise FileNotFoundError("Diretório não encontrado")
else:
    print("Diretório encontrado")


Diretório encontrado


In [8]:
from os import listdir
from os.path import isfile, join

In [9]:
[f for f in listdir(DATA_DIR) if isfile(join(DATA_DIR, f))]

['Damage Propagation Modeling.pdf',
 'readme.txt',
 'RUL_FD001.txt',
 'RUL_FD002.txt',
 'RUL_FD003.txt',
 'RUL_FD004.txt',
 'test_FD001.txt',
 'test_FD002.txt',
 'test_FD003.txt',
 'test_FD004.txt',
 'train_FD001.txt',
 'train_FD002.txt',
 'train_FD003.txt',
 'train_FD004.txt']

#### **3.3 Definição dos nomes das colunas**

Os arquivos originais do C-MAPSS não possuem cabeçalho. Por isso, os nomes das colunas precisam ser definidos manualmente. A estrutura utilizada contém o identificador do motor (`unit`), o ciclo operacional (`cycle`), três condições operacionais (`op1`, `op2` e `op3`) e vinte e um sensores (`s1` a `s21`).


In [10]:
columns = (["unit", "cycle", "op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)])

In [14]:

print(columns)


['unit', 'cycle', 'op1', 'op2', 'op3', 's1', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's20', 's21']


#### **3.4 Carregando um subconjunto do C-MAPSS**

A função abaixo carrega os três arquivos associados a um subconjunto do C-MAPSS: treinamento, teste e RUL verdadeiro do teste. Neste primeiro momento, será utilizado o FD001. A função foi escrita de forma geral para permitir a leitura de outros subconjuntos, se necessário.


In [15]:
def load_cmapss_subset(subset: str, data_dir: Path = DATA_DIR):
    """
    Carrega os arquivos train, test e RUL de um subconjunto C-MAPSS.

    Parâmetros:
        subset: é o identificador do subconjunto, como por exemplo: "FD001"
        data_dir: é o diretório onde estão os arquivos do C-MAPSS

    Retonos:
        train_df: contém os dados de treinamento.
        test_df: contém os dados de teste.
        rul_df: contém os valores reais de RUL para os motores do conjunto de teste.
    """
    train_file = data_dir / f"train_{subset}.txt"
    test_file = data_dir / f"test_{subset}.txt"
    rul_file = data_dir / f"RUL_{subset}.txt"

    required_files = [train_file, test_file, rul_file]
    missing_files = [str(file) for file in required_files if not file.exists()]

    if missing_files:
        raise FileNotFoundError("Os seguinte arquivos a seguir não foram encontrados: ".join(missing_files))

    train_df = pd.read_csv(train_file, sep=r"\s+", header=None, names=columns)
    test_df = pd.read_csv(test_file, sep=r"\s+", header=None, names=columns)
    rul_df = pd.read_csv(rul_file, sep=r"\s+", header=None, names=["RUL"])

    return train_df, test_df, rul_df


#### **3.5 Carregamento do FD001**

O subconjunto FD001 será carregado nesta etapa para iniciar a análise experimental. Após a leitura, são verificadas as dimensões dos arquivos e a quantidade de motores presentes nos conjuntos de treinamento e teste.


In [22]:
train_df, test_df, rul_df = load_cmapss_subset("FD001")

data_info = {
    "Arquivo": ["Treinamento", "Teste", "RUL teste"],
    "Linhas": [train_df.shape[0], test_df.shape[0], rul_df.shape[0]],
    "Colunas": [train_df.shape[1], test_df.shape[1], rul_df.shape[1]],
    "Motores": [train_df["unit"].nunique(), test_df["unit"].nunique(), rul_df.shape[0]]
}

dataset_info = pd.DataFrame(data_info)



In [23]:
dataset_info

,Arquivo,Linhas,Colunas,Motores
0,Treinamento,20631,26,100
1,Teste,13096,26,100
2,RUL teste,100,1,100


#### **3.6 Visualização inicial dos dados**

A visualização das primeiras linhas permite confirmar se os valores foram carregados corretamente e se as colunas estão alinhadas com a estrutura esperada do C-MAPSS.


In [24]:
train_df.head()


,unit,cycle,op1,op2,op3,s1,s2,s3,s4,s5,s6,s7,s8,s9,s10,s11,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,21.61,554.36,2388.06,9046.19,1.3,47.47,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,9044.07,1.3,47.49,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,21.61,554.26,2388.08,9052.94,1.3,47.27,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,2388.11,9049.48,1.3,47.13,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,21.61,554.00,2388.06,9055.15,1.3,47.28,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [25]:
test_df.head()


,unit,cycle,op1,op2,op3,s1,s2,s3,s4,s5,s6,s7,s8,s9,s10,s11,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21
0,1,1,0.0023,0.0003,100.0,518.67,643.02,1585.29,1398.21,14.62,21.61,553.90,2388.04,9050.17,1.3,47.20,521.72,2388.03,8125.55,8.4052,0.03,392,2388,100.0,38.86,23.3735
1,1,2,-0.0027,-0.0003,100.0,518.67,641.71,1588.45,1395.42,14.62,21.61,554.85,2388.01,9054.42,1.3,47.50,522.16,2388.06,8139.62,8.3803,0.03,393,2388,100.0,39.02,23.3916
2,1,3,0.0003,0.0001,100.0,518.67,642.46,1586.94,1401.34,14.62,21.61,554.11,2388.05,9056.96,1.3,47.50,521.97,2388.03,8130.10,8.4441,0.03,393,2388,100.0,39.08,23.4166
3,1,4,0.0042,0.0000,100.0,518.67,642.44,1584.12,1406.42,14.62,21.61,554.07,2388.03,9045.29,1.3,47.28,521.38,2388.05,8132.90,8.3917,0.03,391,2388,100.0,39.00,23.3737
4,1,5,0.0014,0.0000,100.0,518.67,642.51,1587.19,1401.92,14.62,21.61,554.16,2388.01,9044.55,1.3,47.31,522.15,2388.03,8129.54,8.4031,0.03,390,2388,100.0,38.99,23.4130


In [26]:
rul_df.head()


,RUL
0,112
1,98
2,69
3,82
4,91


#### **3.7 Verificações iniciais**

Antes de partir para a construção do RUL e para a análise exploratória, algumas verificações simples são necessárias, como a verificação da quantidade de colunas, da presença de valores ausentes e de ciclos mínimos e máximos por conjunto.


In [27]:
data_checks = {
    "Conjunto": ["Treinamento", "Teste"],
    "Quantidade de colunas": [train_df.shape[1], test_df.shape[1]],
    "Valores ausentes": [train_df.isna().sum().sum(), test_df.isna().sum().sum()],
    "Menor ciclo": [train_df["cycle"].min(), test_df["cycle"].min()],
    "Maior ciclo": [train_df["cycle"].max(), test_df["cycle"].max()],
    "Quantidade de motores": [train_df["unit"].nunique(), test_df["unit"].nunique()]
}

initial_checks = pd.DataFrame(data_checks)


In [28]:
initial_checks

,Conjunto,Quantidade de colunas,Valores ausentes,Menor ciclo,Maior ciclo,Quantidade de motores
0,Treinamento,26,0,1,362,100
1,Teste,26,0,1,303,100


#### **3.8 Interpretação inicial**

Com o carregamento concluído, espera-se que os arquivos de treinamento e teste apresentem a mesma quantidade de colunas, seguindo a estrutura formada por identificação do motor, ciclo, condições operacionais e sensores. O arquivo de RUL, por sua vez, deve conter um valor para cada motor do conjunto de teste.

Essa verificação inicial é necessária porque as próximas etapas dependem diretamente da consistência desses arquivos. Na sequência, será construída a variável RUL para o conjunto de treinamento, utilizando o último ciclo observado de cada motor como referência para a falha.


### **Referências Bibliográficas**

- LIU, Y.; FREDERICK, D. K.; DECASTRO, J. A.; LITT, J. S.; CHAN, W. W. **User’s Guide for the Commercial Modular Aero-Propulsion System Simulation (C-MAPSS), Version 2**. NASA/TM—2012-217432, 2012.
- SAXENA, A.; GOEBEL, K.; SIMON, D.; EKLUND, N. **Damage Propagation Modeling for Aircraft Engine Run-to-Failure Simulation**. Proceedings of the 1st International Conference on Prognostics and Health Management, 2008.
- AVSAR, R. **Enhancing prognostic model interpretability for advanced engine failure prediction using prognostic metrics and explainable AI**. *The Aeronautical Journal*, 2026.
- CHOLA, A.; RASTOGI, R.; KAUR, P.; CHAUDHARY, A.; BISWAS, D. **Predictive Analytics Beyond the Hype: A Comprehensive Comparison of LSTM, XGBoost and LightGBM with Emphasis on RMSE and CPU Utilization**. 2024.
- GOYAL, N.; SACHDEVA, R. **Application of Machine Learning in Predictive Maintenance**. 2025.
- HATIPOĞLU, A.; YILMAZ, E. **A Matrix-Statistics-Aware Attention Mechanism for Robust RUL Estimation in Aero-Engines**. *Applied Sciences*, v. 16, n. 169, 2026.
- JIA, Z.; XIAO, Z.; SHI, Y. **Remaining Useful Life Prediction of Equipment Based on XGBoost**. In: *The 5th International Conference on Computer Science and Application Engineering (CSAE 2021)*, 2021. DOI: 10.1145/3487075.3487134.
- KHALIFEH, A.; ALMEQDADI, S. **Machine Learning-Based Remaining Useful Life Predictions and Its Application on Predictive Maintenance**. In: *2024 22nd International Conference on Research and Education in Mechatronics (REM)*, 2024.
- ZHAN, Y.; WANG, Z.; XU, Z. **Remaining Useful Life Prediction Considering Data and Model Uncertainties**. In: *2023 9th International Symposium on System Security, Safety, and Reliability (ISSSR)*, 2023.
